In [1]:
print ("jupyter is working")


jupyter is working


In [3]:
pip install psycopg2-binary

Defaulting to user installation because normal site-packages is not writeable
     |████████████████████████████████| 3.8 MB 6.8 MB/s eta 0:00:01
You should consider upgrading via the '/Applications/Xcode.app/Contents/Developer/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install requests pandas

Defaulting to user installation because normal site-packages is not writeable
     |████████████████████████████████| 10.8 MB 2.9 MB/s eta 0:00:01
     |████████████████████████████████| 510 kB 16.7 MB/s eta 0:00:01
     |████████████████████████████████| 5.3 MB 5.8 MB/s eta 0:00:01
You should consider upgrading via the '/Applications/Xcode.app/Contents/Developer/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [28]:
from dotenv import load_dotenv
import os
load_dotenv()
API_KEY = os.getenv("API_KEY")
API_HOST = "aerodatabox.p.rapidapi.com"
HEADERS = {
    "x-rapidapi-key": API_KEY,
    "x-rapidapi-host": API_HOST
}
DB_CONFIG = {
    "host": "localhost",
    "port": "5432",
    "database": "air_tracker",
    "user": "postgres",
    "password": os.getenv("DB_PASSWORD")
}
print("Config ready!")

Config ready!


In [6]:
import requests
import pandas as pd
import psycopg2
import json
from datetime import datetime, timedelta
print("All libraries imported successfully!")

All libraries imported successfully!


In [7]:
def get_connection():
    conn = psycopg2.connect(
        host=DB_CONFIG["host"],
        port=DB_CONFIG["port"],
        database=DB_CONFIG["database"],
        user=DB_CONFIG["user"],
        password=DB_CONFIG["password"]
    )
    return conn

# Test the connection
try:
    conn = get_connection()
    print("Connected to PostgreSQL successfully!")
    conn.close()
except Exception as e:
    print(f"Connection failed: {e}")

Connected to PostgreSQL successfully!


In [10]:
import time
AIRPORT_CODES = [
    "DXB", "SIN", "LHR", "JFK", "SYD", "DOH", "BKK", "CDG",
    "MAA", "DEL", "BOM", "BLR", "HYD", "CCU", "COK"
]
def fetch_airport(iata_code):
    url = f"https://{API_HOST}/airports/iata/{iata_code}"
    try:
        response = requests.get(url, headers=HEADERS)
        response.raise_for_status()
        return response.json()
    except Exception as e:
        print(f"Error fetching {iata_code}: {e}")
        return None
airports_data = []
for code in AIRPORT_CODES:
    print(f"Fetching {code}...")
    data = None
    attempts = 0
    while data is None and attempts < 3:
        data = fetch_airport(code)
        if data is None:
            time.sleep(3)
        attempts += 1
    if data:
        airports_data.append(data)
    time.sleep(2)
print(f"\nSuccessfully fetched {len(airports_data)} airports!")

Fetching DXB...
Fetching SIN...
Fetching LHR...
Fetching JFK...
Fetching SYD...
Fetching DOH...
Fetching BKK...
Fetching CDG...
Fetching MAA...
Fetching DEL...
Fetching BOM...
Fetching BLR...
Fetching HYD...
Fetching CCU...
Fetching COK...

Successfully fetched 15 airports!


In [11]:
print(json.dumps(airports_data[0], indent=2))

{
  "icao": "OMDB",
  "iata": "DXB",
  "shortName": "Dubai",
  "fullName": "Dubai",
  "municipalityName": "Dubai",
  "location": {
    "lat": 25.252798,
    "lon": 55.3644
  },
  "elevation": {
    "meter": 18.9,
    "km": 0.02,
    "mile": 0.01,
    "nm": 0.01,
    "feet": 62.0
  },
  "country": {
    "code": "AE",
    "name": "United Arab Emirates"
  },
  "continent": {
    "code": "AS",
    "name": "Asia"
  },
  "timeZone": "Asia/Dubai",
  "urls": {
    "webSite": "http://www.dubaiairports.ae/",
    "wikipedia": "https://en.wikipedia.org/wiki/Dubai_International_Airport",
    "twitter": "https://x.com/DubaiAirports",
    "flightRadar": "https://www.flightradar24.com/25.25,55.36/14",
    "googleMaps": "https://www.google.com/maps/@25.252799,55.364398,14z"
  }
}


In [12]:
# Insert airport data into PostgreSQL
def insert_airports(airports):
    conn = get_connection()
    cursor = conn.cursor()
    
    for airport in airports:
        try:
            cursor.execute("""
                INSERT INTO airport 
                (icao_code, iata_code, name, city, country, continent, latitude, longitude, timezone)
                VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s)
                ON CONFLICT (iata_code) DO NOTHING
            """, (
                airport.get("icao"),
                airport.get("iata"),
                airport.get("fullName"),
                airport.get("municipalityName"),
                airport.get("country", {}).get("name"),
                airport.get("continent", {}).get("name"),
                airport.get("location", {}).get("lat"),
                airport.get("location", {}).get("lon"),
                airport.get("timeZone")
            ))
        except Exception as e:
            print(f"Error inserting {airport.get('iata')}: {e}")
    
    conn.commit()
    cursor.close()
    conn.close()
    print(f"Inserted {len(airports)} airports successfully!")

insert_airports(airports_data)

Inserted 15 airports successfully!


In [14]:
def fetch_flights(iata_code, date):
    url = f"https://{API_HOST}/flights/airports/iata/{iata_code}"
    params = {
        "fromLocal": f"{date}T00:00",
        "toLocal": f"{date}T23:59",
        "withLeg": "true",
        "direction": "Both",
        "withCancelled": "true",
        "withCodeshared": "false",
        "withCargo": "false",
        "withPrivate": "false"
    }
    try:
        response = requests.get(url, headers=HEADERS, params=params)
        response.raise_for_status()
        return response.json()
    except Exception as e:
        print(f"Error fetching flights for {iata_code}: {e}")
        return None

# Test with MAA for yesterday
yesterday = (datetime.now() - timedelta(days=1)).strftime("%Y-%m-%d")
print(f"Fetching flights for MAA on {yesterday}...")
test_flights = fetch_flights("MAA", yesterday)
print(json.dumps(test_flights, indent=2))

Fetching flights for MAA on 2026-06-03...
{
  "departures": [
    {
      "departure": {
        "scheduledTime": {
          "utc": "2026-06-04 00:40Z",
          "local": "2026-06-04 06:10+05:30"
        },
        "terminal": "2",
        "quality": [
          "Basic"
        ]
      },
      "arrival": {
        "airport": {
          "icao": "OBBI",
          "iata": "BAH",
          "name": "Manama",
          "countryCode": "bh",
          "timeZone": "Asia/Bahrain"
        },
        "scheduledTime": {
          "utc": "2026-06-04 05:35Z",
          "local": "2026-06-04 08:35+03:00"
        },
        "quality": [
          "Basic"
        ]
      },
      "number": "GF 69",
      "callSign": "GFA069",
      "status": "Unknown",
      "codeshareStatus": "IsOperator",
      "isCargo": false,
      "aircraft": {
        "reg": "A9C-XB",
        "modeS": "8940CB",
        "model": "Airbus A321 NEO"
      },
      "airline": {
        "name": "Gulf Air",
        "iata": "GF",
    

In [15]:
# Fetch and insert flights for all airports
def insert_flights(flights_list, airport_iata):
    conn = get_connection()
    cursor = conn.cursor()
    inserted = 0
    
    for flight in flights_list:
        try:
            cursor.execute("""
                INSERT INTO flights 
                (flight_id, flight_number, aircraft_model, origin_iata, destination_iata,
                scheduled_departure, actual_departure, scheduled_arrival, actual_arrival,
                status, airline_code)
                VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
                ON CONFLICT (flight_id) DO NOTHING
            """, (
                flight.get("number"),
                flight.get("number"),
                flight.get("aircraft", {}).get("model"),
                flight.get("departure", {}).get("airport", {}).get("iata", airport_iata),
                flight.get("arrival", {}).get("airport", {}).get("iata"),
                flight.get("departure", {}).get("scheduledTime", {}).get("local"),
                flight.get("departure", {}).get("actualTime", {}).get("local"),
                flight.get("arrival", {}).get("scheduledTime", {}).get("local"),
                flight.get("arrival", {}).get("actualTime", {}).get("local"),
                flight.get("status"),
                flight.get("airline", {}).get("iata")
            ))
            inserted += 1
        except Exception as e:
            print(f"Error inserting flight {flight.get('number')}: {e}")
    
    conn.commit()
    cursor.close()
    conn.close()
    return inserted

# Fetch flights for all 15 airports
yesterday = (datetime.now() - timedelta(days=1)).strftime("%Y-%m-%d")
total_flights = 0

for code in AIRPORT_CODES:
    print(f"Fetching flights for {code}...")
    data = fetch_flights(code, yesterday)
    if data:
        flights_list = []
        if "departures" in data:
            flights_list += data["departures"]
        if "arrivals" in data:
            flights_list += data["arrivals"]
        count = insert_flights(flights_list, code)
        total_flights += count
        print(f"  → Inserted {count} flights")
    time.sleep(2)

print(f"\nTotal flights inserted: {total_flights}")

Fetching flights for DXB...
  → Inserted 425 flights
Fetching flights for SIN...
  → Inserted 571 flights
Fetching flights for LHR...
  → Inserted 639 flights
Fetching flights for JFK...
  → Inserted 460 flights
Fetching flights for SYD...
  → Inserted 509 flights
Fetching flights for DOH...
  → Inserted 198 flights
Fetching flights for BKK...
  → Inserted 608 flights
Fetching flights for CDG...
  → Inserted 734 flights
Fetching flights for MAA...
  → Inserted 205 flights
Fetching flights for DEL...
  → Inserted 798 flights
Fetching flights for BOM...
  → Inserted 487 flights
Fetching flights for BLR...
  → Inserted 482 flights
Fetching flights for HYD...
  → Inserted 289 flights
Fetching flights for CCU...
  → Inserted 202 flights
Fetching flights for COK...
  → Inserted 84 flights

Total flights inserted: 6691


In [21]:
# Test with different airline name formats
for name in ["KLM", "indigo", "6E", "IGO", "emirates", "Emirates"]:
    result = fetch_airline_aircraft(name)
    if result:
        print(f"✓ Works with: {name}")
        print(json.dumps(result, indent=2))
        break
    else:
        print(f"✗ Failed with: {name}")
    time.sleep(1)

✓ Works with: KLM
{
  "totalCount": 132,
  "pageOffset": 0,
  "pageSize": 10,
  "hasNextPage": true,
  "count": 10,
  "items": [
    {
      "id": 2156583,
      "reg": "PH-AXR",
      "active": true,
      "serial": "13080",
      "hexIcao": "4868F8",
      "airlineName": "KLM",
      "iataCodeShort": "32Q",
      "icaoCode": "A21N",
      "model": "A21N",
      "modelCode": "321-252NX",
      "numSeats": 227,
      "firstFlightDate": "2026-02-27",
      "deliveryDate": "2026-03-27",
      "registrationDate": "2026-03-27",
      "typeName": "Airbus A321 NEO",
      "numEngines": 2,
      "engineType": "Jet",
      "isFreighter": false,
      "productionLine": "Airbus A321 NEO",
      "ageYears": 0.3,
      "verified": true,
      "numRegistrations": 1,
      "registrations": [
        {
          "reg": "PH-AXR",
          "active": true,
          "hexIcao": "4868F8",
          "airlineName": "KLM",
          "registrationDate": "2026-03-27"
        }
      ]
    },
    {
      "id":

In [24]:
# Use IATA codes directly for aircraft fetch
TOP_AIRLINES = ["6E", "AF", "AI", "BA", "QF", "EK", "IX", "SQ", "QR", "DL"]

total_aircraft = 0
for code in TOP_AIRLINES:
    print(f"Fetching fleet for {code}...")
    data = fetch_airline_aircraft(code)
    if data and "items" in data:
        count = insert_aircraft(data["items"], code)
        total_aircraft += count
        print(f"  → Inserted {count} aircraft")
    else:
        print(f"  → No data found")
    time.sleep(2)

print(f"\nTotal aircraft inserted: {total_aircraft}")

Fetching fleet for 6E...
  → Inserted 10 aircraft
Fetching fleet for AF...
  → Inserted 10 aircraft
Fetching fleet for AI...
  → Inserted 10 aircraft
Fetching fleet for BA...
  → Inserted 10 aircraft
Fetching fleet for QF...
  → Inserted 10 aircraft
Fetching fleet for EK...
  → Inserted 10 aircraft
Fetching fleet for IX...
  → Inserted 10 aircraft
Fetching fleet for SQ...
  → Inserted 10 aircraft
Fetching fleet for QR...
  → Inserted 10 aircraft
Fetching fleet for DL...
  → Inserted 10 aircraft

Total aircraft inserted: 100


In [26]:
# Test airport delays with ICAO code
def fetch_airport_delays(icao_code):
    url = f"https://{API_HOST}/airports/icao/{icao_code}/delays"
    try:
        response = requests.get(url, headers=HEADERS)
        response.raise_for_status()
        return response.json()
    except Exception as e:
        print(f"Error fetching delays for {icao_code}: {e}")
        return None

# Test with MAA (ICAO: VOMM)
test_delays = fetch_airport_delays("VOMM")
print(json.dumps(test_delays, indent=2))

{
  "airportIcao": "VOMM",
  "from": {
    "utc": "2026-06-04 07:09Z",
    "local": "2026-06-04 07:09+00:00"
  },
  "to": {
    "utc": "2026-06-04 09:09Z",
    "local": "2026-06-04 09:09+00:00"
  },
  "departuresDelayInformation": {
    "numTotal": 17,
    "numQualifiedTotal": 0,
    "numCancelled": 0
  },
  "arrivalsDelayInformation": {
    "numTotal": 19,
    "numQualifiedTotal": 0,
    "numCancelled": 0
  }
}


In [27]:
# Airport ICAO to IATA mapping
AIRPORT_ICAO = {
    "DXB": "OMDB", "SIN": "WSSS", "LHR": "EGLL", "JFK": "KJFK",
    "SYD": "YSSY", "DOH": "OTHH", "BKK": "VTBS", "CDG": "LFPG",
    "MAA": "VOMM", "DEL": "VIDP", "BOM": "VABB", "BLR": "VOBL",
    "HYD": "VOHS", "CCU": "VECC", "COK": "VOCI"
}

def insert_delays(iata_code, data):
    conn = get_connection()
    cursor = conn.cursor()
    try:
        total = (data.get("departuresDelayInformation", {}).get("numTotal", 0) +
                 data.get("arrivalsDelayInformation", {}).get("numTotal", 0))
        delayed = (data.get("departuresDelayInformation", {}).get("numQualifiedTotal", 0) +
                   data.get("arrivalsDelayInformation", {}).get("numQualifiedTotal", 0))
        cancelled = (data.get("departuresDelayInformation", {}).get("numCancelled", 0) +
                     data.get("arrivalsDelayInformation", {}).get("numCancelled", 0))
        cursor.execute("""
            INSERT INTO airport_delays 
            (airport_iata, delay_date, total_flights, delayed_flights, avg_delay_min, median_delay_min, canceled_flights)
            VALUES (%s, %s, %s, %s, %s, %s, %s)
        """, (
            iata_code,
            data.get("from", {}).get("utc"),
            total,
            delayed,
            0,
            0,
            cancelled
        ))
        conn.commit()
        print(f"  → Inserted delays for {iata_code}")
    except Exception as e:
        print(f"Error inserting delays for {iata_code}: {e}")
    cursor.close()
    conn.close()

# Fetch delays for all 15 airports
for iata, icao in AIRPORT_ICAO.items():
    print(f"Fetching delays for {iata}...")
    data = fetch_airport_delays(icao)
    if data:
        insert_delays(iata, data)
    time.sleep(2)

print("\nAll delays inserted!")

Fetching delays for DXB...
  → Inserted delays for DXB
Fetching delays for SIN...
  → Inserted delays for SIN
Fetching delays for LHR...
  → Inserted delays for LHR
Fetching delays for JFK...
  → Inserted delays for JFK
Fetching delays for SYD...
  → Inserted delays for SYD
Fetching delays for DOH...
  → Inserted delays for DOH
Fetching delays for BKK...
  → Inserted delays for BKK
Fetching delays for CDG...
  → Inserted delays for CDG
Fetching delays for MAA...
  → Inserted delays for MAA
Fetching delays for DEL...
  → Inserted delays for DEL
Fetching delays for BOM...
  → Inserted delays for BOM
Fetching delays for BLR...
  → Inserted delays for BLR
Fetching delays for HYD...
Error fetching delays for VOHS: 400 Client Error: Bad Request for url: https://aerodatabox.p.rapidapi.com/airports/icao/VOHS/delays
Fetching delays for CCU...
Error fetching delays for VECC: 400 Client Error: Bad Request for url: https://aerodatabox.p.rapidapi.com/airports/icao/VECC/delays
Fetching delays for CO